# pems05 按年节点增长数据处理

规则：
- 扫描 pems05 目录下 `d05_text_station_5min_YYYY_MM_DD.txt.gz`。
- **首年**：以数据集**首日**的 stations 作为该年的固定节点集合。
- **其余年份**：以每年 **1月1日** 当天的 stations 作为该年的节点集合。
- 每年只保留「该年锚定日」出现的节点，每年保存为一个 CSV。

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
from collections import defaultdict

In [2]:
PEMS_DIR = './pems05'
OUTPUT_DIR = '.'
PREFIX = 'd05_text_station_5min'
SUFFIX = '.txt.gz'

COLUMN_NAMES = [
    "Timestamp", "Station", "District", "Freeway #",
    "Direction of Travel", "Lane Type", "Station Length",
    "Samples", "% Observed", "Total Flow", "Avg Occupancy", "Avg Speed"
]

## 1. 读取 pems05 目录，解析所有可用日期

In [3]:
def list_pems_dates(data_dir):
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"目录不存在: {data_dir}")
    dates = {}
    for f in os.listdir(data_dir):
        if not f.startswith(PREFIX) or not f.endswith(SUFFIX):
            continue
        try:
            rest = f[len(PREFIX):-len(SUFFIX)].strip(' _')
            parts = rest.split('_')
            if len(parts) == 3:
                y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
                dt = date(y, m, d)
                dates[dt] = os.path.join(data_dir, f)
        except (ValueError, IndexError):
            continue
    return dates

available = list_pems_dates(PEMS_DIR)
sorted_dates = sorted(available.keys())
first_date = min(sorted_dates)
print(f"pems05 共 {len(available)} 个日期文件")
print(f"首日: {first_date}, 末日: {sorted_dates[-1]}")

pems05 共 7586 个日期文件
首日: 2005-03-24, 末日: 2025-12-31


## 2. 确定每年的锚定日与节点集合

In [4]:
def get_anchor_date(year, first_date):
    """首年用数据集首日，其余年份用该年 1 月 1 日。"""
    if year == first_date.year:
        return first_date
    return date(year, 1, 1)

def read_station_ids_from_file(path):
    df = pd.read_csv(path, header=None, usecols=[1], names=['Station'], compression='gzip')
    return np.sort(df['Station'].dropna().unique().astype(int))

years = sorted(set(d.year for d in sorted_dates))
year_anchor = {}
year_nodes = {}

for y in years:
    anchor = get_anchor_date(y, first_date)
    year_anchor[y] = anchor
    if anchor not in available:
        print(f"警告: {y} 年锚定日 {anchor} 无数据文件，跳过该年")
        continue
    path = available[anchor]
    year_nodes[y] = read_station_ids_from_file(path)
    print(f"{y} 年 锚定日={anchor}, 节点数={len(year_nodes[y])}")

2005 年 锚定日=2005-03-24, 节点数=6
2006 年 锚定日=2006-01-01, 节点数=8
2007 年 锚定日=2007-01-01, 节点数=10
2008 年 锚定日=2008-01-01, 节点数=10
2009 年 锚定日=2009-01-01, 节点数=10
2010 年 锚定日=2010-01-01, 节点数=10
2011 年 锚定日=2011-01-01, 节点数=10
2012 年 锚定日=2012-01-01, 节点数=10
2013 年 锚定日=2013-01-01, 节点数=153
2014 年 锚定日=2014-01-01, 节点数=153
2015 年 锚定日=2015-01-01, 节点数=153
2016 年 锚定日=2016-01-01, 节点数=153
2017 年 锚定日=2017-01-01, 节点数=202
2018 年 锚定日=2018-01-01, 节点数=243
2019 年 锚定日=2019-01-01, 节点数=382
2020 年 锚定日=2020-01-01, 节点数=485
2021 年 锚定日=2021-01-01, 节点数=486
2022 年 锚定日=2022-01-01, 节点数=515
2023 年 锚定日=2023-01-01, 节点数=568
2024 年 锚定日=2024-01-01, 节点数=568
2025 年 锚定日=2025-01-01, 节点数=572


## 3. 按年汇总：只保留锚定日节点，每年保存一个 CSV

In [5]:
def read_one_day(path, columns=None):
    if columns is None:
        columns = list(range(12))
    df = pd.read_csv(path, header=None, usecols=columns,
                    names=[COLUMN_NAMES[i] for i in columns], compression='gzip')
    return df


def generate_empty_day_df(day, node_set):
    """当某天文件缺失时，为该天所有节点生成 5 分钟分辨率、全 0 的占位数据。"""
    # PeMS 5min: 24*60/5 = 288 个时间点
    start_dt = datetime(day.year, day.month, day.day, 0, 0)
    timestamps = [
        (start_dt + timedelta(minutes=5 * i)).strftime("%m/%d/%Y %H:%M:%S")
        for i in range(288)
    ]

    rows = []
    for ts in timestamps:
        for st in node_set:
            rows.append([
                ts,          # Timestamp
                st,          # Station
                np.nan,      # District
                np.nan,      # Freeway #
                np.nan,      # Direction of Travel
                np.nan,      # Lane Type
                np.nan,      # Station Length
                0,           # Samples
                0,           # % Observed
                0,           # Total Flow
                0,           # Avg Occupancy
                0,           # Avg Speed
            ])

    return pd.DataFrame(rows, columns=COLUMN_NAMES)


os.makedirs(OUTPUT_DIR, exist_ok=True)

for year in years:
    if year not in year_nodes:
        continue
    node_set = set(year_nodes[year])
    start = first_date if year == first_date.year else date(year, 1, 1)
    end = date(year, 12, 31)
    chunks = []
    current = start
    while current <= end:
        if current in available:
            path = available[current]
            df = read_one_day(path)
            df = df[df['Station'].isin(node_set)]
        else:
            # 该天文件缺失，补 0
            df = generate_empty_day_df(current, node_set)
        chunks.append(df)
        current += timedelta(days=1)
    if not chunks:
        print(f"{year} 年无可用数据，跳过")
        continue
    out_df = pd.concat(chunks, ignore_index=True)
    out_path = os.path.join(OUTPUT_DIR, f'pems05_{year}.csv')
    out_df.to_csv(out_path, index=False)
    print(f"{year}: 节点数={len(node_set)}, 行数={len(out_df)}, 已保存 -> {out_path}")

2005: 节点数=6, 行数=488880, 已保存 -> .\pems05_2005.csv
2006: 节点数=8, 行数=840768, 已保存 -> .\pems05_2006.csv
2007: 节点数=10, 行数=1050960, 已保存 -> .\pems05_2007.csv
2008: 节点数=10, 行数=1053840, 已保存 -> .\pems05_2008.csv
2009: 节点数=10, 行数=1050960, 已保存 -> .\pems05_2009.csv
2010: 节点数=10, 行数=1050960, 已保存 -> .\pems05_2010.csv
2011: 节点数=10, 行数=1050960, 已保存 -> .\pems05_2011.csv
2012: 节点数=10, 行数=161280, 已保存 -> .\pems05_2012.csv
2013: 节点数=153, 行数=16065918, 已保存 -> .\pems05_2013.csv
2014: 节点数=153, 行数=16081524, 已保存 -> .\pems05_2014.csv
2015: 节点数=153, 行数=16032564, 已保存 -> .\pems05_2015.csv
2016: 节点数=153, 行数=12758175, 已保存 -> .\pems05_2016.csv
2017: 节点数=202, 行数=21231815, 已保存 -> .\pems05_2017.csv
2018: 节点数=243, 行数=25541244, 已保存 -> .\pems05_2018.csv
2019: 节点数=382, 行数=39238487, 已保存 -> .\pems05_2019.csv
2020: 节点数=485, 行数=50978194, 已保存 -> .\pems05_2020.csv
2021: 节点数=486, 行数=51001430, 已保存 -> .\pems05_2021.csv
2022: 节点数=515, 行数=53692847, 已保存 -> .\pems05_2022.csv
2023: 节点数=568, 行数=59463714, 已保存 -> .\pems05_2023.csv
2024: 节点数=568,

## 4. 各年节点数汇总

In [ ]:
summary = pd.DataFrame({
    'year': list(year_nodes.keys()),
    'anchor_date': [year_anchor[y] for y in year_nodes],
    'n_nodes': [len(year_nodes[y]) for y in year_nodes],
}).sort_values('year').reset_index(drop=True)
summary

,year,anchor_date,n_nodes
0,2005,2005-03-24,6
1,2006,2006-01-01,8
2,2007,2007-01-01,10
3,2008,2008-01-01,10
4,2009,2009-01-01,10
5,2010,2010-01-01,10
6,2011,2011-01-01,10
7,2012,2012-01-01,10
8,2013,2013-01-01,153
9,2014,2014-01-01,153


: 

## 5. Pivot 处理：行=5min时间步, 列=Station, 值=Total Flow

读取每年的 `pems05_{year}.csv`，pivot 成：
- **行**：完整的 5 分钟时间序列（缺失时间步自动补 NaN）
- **列**：每个 Station ID
- **值**：Total Flow

输出保存为 `pems05_{year}_flow.csv`

In [1]:
import os, re, glob
import pandas as pd
from datetime import date

FLOW_OUTPUT_DIR = '.'
FIRST_DATE = date(2005, 3, 24)
DISTRICT = 'pems05'

_csv_files = glob.glob(os.path.join(FLOW_OUTPUT_DIR, f'{DISTRICT}_[0-9][0-9][0-9][0-9].csv'))
years = sorted(int(re.search(rf'{DISTRICT}_(\d{{4}})\.csv', f).group(1)) for f in _csv_files)
print(f"扫描到 {len(years)} 个年度 CSV: {years[0]}~{years[-1]}")

def pivot_yearly_csv(year, first_date=FIRST_DATE, output_dir=FLOW_OUTPUT_DIR):
    src_path = os.path.join(output_dir, f'{DISTRICT}_{year}.csv')
    if not os.path.exists(src_path):
        print(f"  [跳过] {src_path} 不存在")
        return

    df = pd.read_csv(src_path, usecols=['Timestamp', 'Station', 'Total Flow'])
    print(f"  读取完成: {len(df):,} 行")

    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    pivot = df.pivot_table(
        index='Timestamp', columns='Station',
        values='Total Flow', aggfunc='sum'
    )
    del df

    pivot.columns = pivot.columns.astype(int)
    pivot = pivot[sorted(pivot.columns)]

    if year == first_date.year:
        start_dt = pd.Timestamp(first_date)
    else:
        start_dt = pd.Timestamp(year, 1, 1)
    end_dt = pd.Timestamp(year, 12, 31, 23, 55)

    full_idx = pd.date_range(start=start_dt, end=end_dt, freq='5min')
    pivot = pivot.reindex(full_idx)
    pivot.index.name = 'Timestamp'

    out_path = os.path.join(output_dir, f'{DISTRICT}_{year}_flow.csv')
    pivot.to_csv(out_path)

    n_nan_rows = pivot.isna().all(axis=1).sum()
    print(f"  {year}: stations={pivot.shape[1]}, "
          f"时间步={pivot.shape[0]}(期望{len(full_idx)}), "
          f"全NaN行={n_nan_rows}, 已保存 -> {out_path}")
    return pivot.shape

print("pivot_yearly_csv 函数已定义")

扫描到 21 个年度 CSV: 2005~2025
pivot_yearly_csv 函数已定义


In [2]:
import time

results = {}
for year in years:
    print(f"\n{'='*60}")
    print(f"处理 {year} 年 ...")
    t0 = time.time()
    shape = pivot_yearly_csv(year)
    elapsed = time.time() - t0
    print(f"  耗时: {elapsed:.1f}s")
    if shape:
        results[year] = shape

print(f"\n{'='*60}")
print(f"全部完成! 共处理 {len(results)} 年")


处理 2005 年 ...
  读取完成: 488,880 行
  2005: stations=6, 时间步=81504(期望81504), 全NaN行=24, 已保存 -> .\pems05_2005_flow.csv
  耗时: 0.7s

处理 2006 年 ...
  读取完成: 840,768 行
  2006: stations=8, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems05_2006_flow.csv
  耗时: 1.0s

处理 2007 年 ...
  读取完成: 1,050,960 行
  2007: stations=10, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems05_2007_flow.csv
  耗时: 1.3s

处理 2008 年 ...
  读取完成: 1,053,840 行
  2008: stations=10, 时间步=105408(期望105408), 全NaN行=24, 已保存 -> .\pems05_2008_flow.csv
  耗时: 1.2s

处理 2009 年 ...
  读取完成: 1,050,960 行
  2009: stations=10, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems05_2009_flow.csv
  耗时: 1.2s

处理 2010 年 ...
  读取完成: 1,050,960 行
  2010: stations=10, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems05_2010_flow.csv
  耗时: 1.2s

处理 2011 年 ...
  读取完成: 1,050,960 行
  2011: stations=10, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems05_2011_flow.csv
  耗时: 1.3s

处理 2012 年 ...
  读取完成: 161,280 行
  2012: stations=10, 时间步=105408(期望105408), 全NaN行=89280, 已保存 -> .\pems05_

## 6. 检查输出结果汇总

In [3]:
summary_flow = pd.DataFrame([
    {'year': y, 'rows': s[0], 'stations': s[1]}
    for y, s in results.items()
])
summary_flow = summary_flow.sort_values('year').reset_index(drop=True)
print("各年 flow pivot 结果汇总:")
summary_flow

各年 flow pivot 结果汇总:


,year,rows,stations
0,2005,81504,6
1,2006,105120,8
2,2007,105120,10
3,2008,105408,10
4,2009,105120,10
5,2010,105120,10
6,2011,105120,10
7,2012,105408,10
8,2013,105120,153
9,2014,105120,153


In [4]:
check = pd.read_csv(f'./{DISTRICT}_{years[0]}_flow.csv', index_col=0, parse_dates=True, nrows=10)
print(f"{DISTRICT}_{years[0]}_flow.csv 前 10 行, shape={check.shape}")
check

pems05_2005_flow.csv 前 10 行, shape=(10, 6)


,500001,500003,500004,500005,500006,500007
Timestamp,,,,,,
2005-03-24 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0
2005-03-24 00:05:00,0.0,0.0,0.0,0.0,0.0,0.0
2005-03-24 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0
2005-03-24 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0
2005-03-24 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0
2005-03-24 00:25:00,0.0,0.0,0.0,0.0,0.0,0.0
2005-03-24 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0
2005-03-24 00:35:00,0.0,0.0,0.0,0.0,0.0,0.0
2005-03-24 00:40:00,0.0,0.0,0.0,0.0,0.0,0.0
